In [ ]:
import numpy as np
import pandas as pd

import yfinance as yf
from fredapi import Fred

import os

import matplotlib.pyplot as plt
import seaborn as sns

# MacroEconomic Data

In [ ]:
fred_api_key = '7e7047cb386c073a68b26832503fa819'

start = '1990-01-01'
end = '2025-12-31'

In [ ]:
macro_variables = [
    'INDPRO', 'W875RX1', 'UNRATE', 'ICSA', 'CPIAUCSL', 'DCOILWTICO', 'VIXCLS',
    'BAA10Y', 'T10Y2Y', 'FEDFUNDS', 'UMCSENT', 'HOUST', 'M2SL'
]

In [ ]:
fred = Fred(api_key=fred_api_key)

In [ ]:
series_dict = {}

for var in macro_variables:
    series_dict[var] = fred.get_series(var, observation_start=start, observation_end=end)

df_raw = pd.DataFrame(series_dict)

In [ ]:
df_raw.head(10)

In [ ]:
df_monthly  = df_raw.resample('ME').last()

In [ ]:
df_monthly.head(10)

In [ ]:
df_monthly.isna().sum()

In [ ]:
df_monthly['UNRATE'] = df_monthly['UNRATE'].ffill()
df_monthly['CPIAUCSL'] = df_monthly['CPIAUCSL'].ffill()

In [ ]:
macro_var = {
    #Real Activity
    "INDPRO": "industrial_production",
    "W875RX1": "real_person_income",

    #Labour Market
    "UNRATE": "unemployment_rate",
    "ICSA": "initial_jobless_claims",

    #Inflation
    "CPIAUCSL": "cpi",
    "DCOILWTICO": "oil_price",

    #Financial Condition
    "VIXCLS": "vix",
    "BAA10Y": "credit_spread",

    #Monetary Policy
    "T10Y2Y": "yield_curve_slope",
    "FEDFUNDS": "fed_funds_rate",

    #Sentiment & Housing
    "UMCSENT": "consumer_sentiment",
    "HOUST": "housing_starts",
    "M2SL": "m2_money_supply"
}

In [ ]:
df = df_monthly.rename(columns=macro_var)

In [ ]:
df.head()

In [ ]:
df.to_csv('../data/macro_data.csv')

# Market Data

In [ ]:
tickers = ['VUSTX', 'VFINX']

In [ ]:
asset = yf.download(tickers=tickers, start='1990-01-01', end='2025-12-31')['Close']

In [ ]:
gld = pd.read_excel('../data/Gold_price_averages_in_a range_of_currencies_since_1978.xlsx',
                    sheet_name="Monthly_Avg", index_col="Date")

In [ ]:
asset_data = asset.join(gld, how="left")

In [ ]:
asset_data = asset_data.resample('ME').last()

In [ ]:
market_var = {
    #Equity
    "VFINX": "index_fund",

    #Bonds
    "VUSTX": "treasury_fund",

    #Gold
    "USD": "gold_fund"
}

In [ ]:
asset_data = asset_data.rename(columns=market_var)

In [ ]:
asset_data.isna().sum()

In [ ]:
asset_data['gold_fund'] = asset_data['gold_fund'].ffill()

In [ ]:
asset_data.to_csv("../data/market_data.csv")